In [1]:
import sys
sys.path.append('..')
from src.preprocessing import load_processed_data, split_data
from src.modeling import (train_baseline_models, tune_random_forest, tune_xgboost,
                          tune_lightgbm, train_catboost_default, save_model)
import pandas as pd
import numpy as np

In [2]:
X, y = load_processed_data('../data/processed')
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)

In [3]:
rf_model, rf_params = tune_random_forest(X_train, y_train, X_val, y_val)
xgb_model, xgb_params = tune_xgboost(X_train, y_train, X_val, y_val)
lgb_model, lgb_params = tune_lightgbm(X_train, y_train, X_val, y_val)
cat_model = train_catboost_default(X_train, y_train, X_val, y_val)

RF tuned RMSE: 233.37, best params: {'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 10}
XGBoost tuned RMSE: 226.99, best params: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}
LightGBM tuned RMSE: 226.19, best params: {'num_leaves': 31, 'n_estimators': 100, 'max_depth': 20, 'learning_rate': 0.1}
CatBoost default RMSE: 225.10


In [4]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def get_metrics(model, X_val, y_val):
    pred = model.predict(X_val)
    return {'RMSE': np.sqrt(mean_squared_error(y_val, pred)),
            'MAE': mean_absolute_error(y_val, pred),
            'R2': r2_score(y_val, pred)}

results = []
results.append({'name': 'RF tuned', **get_metrics(rf_model, X_val, y_val)})
results.append({'name': 'XGBoost tuned', **get_metrics(xgb_model, X_val, y_val)})
results.append({'name': 'LightGBM tuned', **get_metrics(lgb_model, X_val, y_val)})
results.append({'name': 'CatBoost default', **get_metrics(cat_model, X_val, y_val)})

df_results = pd.DataFrame(results).sort_values('RMSE')
print(df_results)

               name        RMSE         MAE        R2
3  CatBoost default  225.103129  151.461374  0.939303
2    LightGBM tuned  226.188506  150.817490  0.938717
1     XGBoost tuned  226.988572  150.263485  0.938282
0          RF tuned  233.366874  152.967171  0.934765


In [14]:
best_model = cat_model
y_pred_test = best_model.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
print(f"Test RMSE: {rmse_test:.2f}")

Test RMSE: 224.93


In [15]:
save_model(best_model, '../models/best_model.pkl')

Модель сохранена в ../models/best_model.pkl
